# Holy Wells in Ireland and the UK — dedications by gender

This notebook queries [Wikidata](https://www.wikidata.org/) for holy
wells in Ireland and the United Kingdom, joins each well to the
saint it is dedicated to, and splits the result by the patron
saint's gender. Two maps side by side make the geography visible;
two more maps (one per gender) colour each well by its specific
patron saint; two bar charts show which individual saints dominate
each group.

It is a companion to the overview notebook
`wikidata-holy-wells-map.ipynb` (marker map + density grid), and
takes one further step: instead of treating all wells as equivalent
points, we ask *whom* each well commemorates and let that shape the
picture.

A browser-executable companion (`wikidata-holy-wells-dedications-live.qmd`)
runs the same pipeline directly in the browser via Pyodide.

## Requirements

```bash
pip install SPARQLWrapper pandas folium matplotlib
```


## About this notebook

### Why this dataset?

Dedications are the most information-dense single property of a
holy well: a well is more than a point on a map, it is a point
*named for* someone. Several features make this slice teachable:

- **A second hop into Wikidata** — the query joins the well
  (`Q1371047`) to the saint (`?namedAfter`) to the saint's gender
  (`P21`). That second hop is where the knowledge graph earns its
  keep; a flat list of wells would not tell you this, but Wikidata
  already knows each saint's biographical data.
- **Sharp, visible asymmetry** — the Irish sacred landscape is
  heavily female-dedicated (Mary, Brigid, Íte), and the surprise of
  *which* saints dominate on a map is a quick hook for discussing
  sample bias, gendered historiography, and the limits of the
  dataset.
- **Two countries, very uneven coverage** — Ireland has hundreds of
  modelled wells, the UK has a handful. This is itself a finding
  about data completeness rather than about holy wells in the
  world.

### Data-context notes

- The query returns **one row per well–patron pair**. Wells with
  multiple patrons produce multiple rows. The grouping step below
  keeps each well once, attaching the list of patrons.
- `P21` (*sex or gender*) on a saint can be either `Q6581072`
  (female) or `Q6581097` (male); these are the two values filtered
  by the `VALUES ?gender { ... }` clause. Other values exist in
  Wikidata for living people but are not relevant here.
- A handful of wells have patrons of both genders (e.g. a joint
  dedication). These appear on **both** overview maps, and on the
  per-saint maps they appear once per patron.
- The UK sample is small enough that a single well (St Winifred's
  Well in Holywell, Wales) is a visible outlier on the female map.

### Tooling notes

Same stack as the overview notebook: `SPARQLWrapper` for the query,
`folium` for the maps (including the per-saint layer toggle via
`folium.FeatureGroup`), and `matplotlib` for the bar charts.
Setting a descriptive `User-Agent` is not optional for Wikidata —
the service rate-limits requests with a generic UA and may block
repeated anonymous access.


## Step 1 — Define the SPARQL query

The query walks two edges of the graph: `?item → ?namedAfter` via
`P138` (*named after*), and `?namedAfter → ?gender` via `P21` (*sex
or gender*). The two `VALUES` clauses filter countries (Ireland,
UK) and genders (female, male) — both are whitelists rather than
`FILTER` expressions, which Wikidata's query planner handles more
efficiently.


In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"
USER_AGENT = "OER-HolyWells-Dedications-Notebook/1.0"

SPARQL_QUERY = """
SELECT DISTINCT ?item ?itemLabel ?namedAfter ?namedAfterLabel
                ?gender ?country ?countryLabel ?geo WHERE {
  ?item wdt:P31 wd:Q1371047;
        wdt:P625 ?geo;
        wdt:P138 ?namedAfter;
        wdt:P17 ?country.
  VALUES ?country { wd:Q27 wd:Q145 }
  ?namedAfter wdt:P21 ?gender.
  VALUES ?gender { wd:Q6581072 wd:Q6581097 }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
"""

sparql = SPARQLWrapper(SPARQL_ENDPOINT, agent=USER_AGENT)
sparql.setQuery(SPARQL_QUERY)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
bindings = results["results"]["bindings"]
print(f"✓ Retrieved {len(bindings)} rows from Wikidata")

## Step 2 — Load the data

Each row is a (well, patron, gender) triple — so wells with multiple
patrons appear multiple times. We build a long DataFrame first
(one row per pair, which is what the bar charts and per-saint maps
want), then derive a wide per-well view (one row per well, patrons
and genders as lists) which is what the gender-overview map wants.


In [ ]:
import re
import pandas as pd

assert bindings, "⚠ Please run the query cell first."

_WKT_POINT_RE = re.compile(
    r"point\s*\(\s*([-+\d.eE]+)\s+([-+\d.eE]+)\s*\)", re.IGNORECASE
)
_GENDER_LABEL = {
    "http://www.wikidata.org/entity/Q6581072": "female",
    "http://www.wikidata.org/entity/Q6581097": "male",
}

def parse_wkt_point(wkt):
    if not wkt:
        return (None, None)
    m = _WKT_POINT_RE.search(wkt)
    if not m:
        return (None, None)
    try:
        lon, lat = float(m.group(1)), float(m.group(2))
        return (lat, lon)
    except ValueError:
        return (None, None)

# --- Long format: one row per well-patron pair ---
long_rows = []
for b in bindings:
    lat, lon = parse_wkt_point(b.get("geo", {}).get("value"))
    long_rows.append({
        "item": b["item"]["value"],
        "qid": b["item"]["value"].rsplit("/", 1)[-1],
        "itemLabel": b.get("itemLabel", {}).get("value", ""),
        "namedAfter": b["namedAfter"]["value"],
        "namedAfterLabel": b.get("namedAfterLabel", {}).get("value", ""),
        "gender": _GENDER_LABEL.get(b["gender"]["value"], "other"),
        "country": b.get("countryLabel", {}).get("value", ""),
        "latitude": lat,
        "longitude": lon,
    })
long_df = pd.DataFrame(long_rows)

# --- Wide format: one row per well, patrons/genders as lists ---
wide_df = (long_df.groupby("item")
                  .agg(itemLabel=("itemLabel", "first"),
                       qid=("qid", "first"),
                       latitude=("latitude", "first"),
                       longitude=("longitude", "first"),
                       country=("country", "first"),
                       patrons=("namedAfterLabel", lambda s: sorted(set(s))),
                       genders=("gender", lambda s: sorted(set(s))))
                  .reset_index())

print(f"✓ {len(long_df)} well–patron pairs")
print(f"  {len(wide_df)} unique wells")
print(f"  country breakdown: {wide_df['country'].value_counts().to_dict()}")
print(f"  wells with patrons of both genders: "
      f"{(wide_df['genders'].apply(len) > 1).sum()}")

wide_df.head()

## Step 3 — Visualise

### Step 3a — Two maps side by side: female and male patrons

Two folium maps, each fed a filtered subset of the wells. A well
with patrons of both genders appears on both maps. Click any marker
for the well's name, its patrons, and a link to its Wikidata item.
Both maps carry a fullscreen control.

The side-by-side layout uses `branca.Figure` so the two maps render
next to each other in the notebook output.


In [ ]:
import folium
from folium.plugins import Fullscreen
from branca.element import Figure

assert 'long_df' in dir() and not long_df.empty, "⚠ Please run the data-loading cell first."

geo_long = long_df[long_df["latitude"].notna()].copy()

# Build one marker record per (well, gender)
def build_gender_markers(gender):
    subset = geo_long[geo_long["gender"] == gender]
    records = {}
    for _, row in subset.iterrows():
        uri = row["item"]
        if uri not in records:
            records[uri] = {
                "lat": float(row["latitude"]),
                "lon": float(row["longitude"]),
                "name": row["itemLabel"],
                "qid": row["qid"],
                "country": row["country"],
                "patrons": [],
            }
        records[uri]["patrons"].append(row["namedAfterLabel"])
    for r in records.values():
        r["patrons"] = sorted(set(r["patrons"]))
    return list(records.values())

female_markers = build_gender_markers("female")
male_markers   = build_gender_markers("male")

print(f"  {len(female_markers)} wells with female patrons")
print(f"  {len(male_markers)} wells with male patrons")

centre = [geo_long["latitude"].mean(), geo_long["longitude"].mean()]

def build_gender_map(markers, colour):
    m = folium.Map(location=centre, zoom_start=6, tiles="OpenStreetMap",
                   width="100%", height="100%")
    Fullscreen(position="topleft").add_to(m)
    for rec in markers:
        popup_html = (
            f'<b>{rec["name"]}</b><br>'
            f'<small>{rec["country"]}</small><br>'
            f'<i>named after</i>: {", ".join(rec["patrons"])}<br>'
            f'<small><a href="https://www.wikidata.org/wiki/{rec["qid"]}" '
            f'target="_blank">Wikidata ({rec["qid"]})</a></small>'
        )
        folium.CircleMarker(
            location=[rec["lat"], rec["lon"]],
            radius=5, color=colour, weight=1,
            fill=True, fill_color=colour, fill_opacity=0.75,
            popup=folium.Popup(popup_html, max_width=260),
        ).add_to(m)
    return m

map_female = build_gender_map(female_markers, "#c2185b")
map_male   = build_gender_map(male_markers,   "#1565c0")

# Side-by-side layout via branca.Figure with two subfigures
fig = Figure(width="100%", height=520)
fig.html.add_child(folium.Element(
    '<h4 style="margin:0 0 4px 0;font:600 14px/1.3 sans-serif">'
    f'<span style="color:#c2185b">●</span> female patrons '
    f'<small>({len(female_markers)} wells)</small>'
    '&nbsp;&nbsp;&nbsp;&nbsp;'
    '<span style="color:#1565c0">●</span> male patrons '
    f'<small>({len(male_markers)} wells)</small>'
    '</h4>'
))
sub_left  = fig.add_subplot(1, 2, 1)
sub_right = fig.add_subplot(1, 2, 2)
sub_left.add_child(map_female)
sub_right.add_child(map_male)
fig

### Step 3b — Female patrons, one layer per saint

Same data as the left map above, but now each saint gets her own
toggleable `folium.FeatureGroup` with a distinct colour — the colour
swatch sits directly in the layer-control label, so the panel itself
acts as the legend. This mirrors the
[Wikidata Query Service map view](https://w.wiki/AQ2F), which uses
the `?layer` variable to drive categorical colouring, and lets you
isolate individual dedications to see their geography on their own.


In [ ]:
assert 'long_df' in dir() and not long_df.empty, "⚠ Please run the data-loading cell first."

# A 24-colour categorical palette — D3's category20 extended.
PALETTE = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b",
    "#e377c2", "#17becf", "#bcbd22", "#7f7f7f", "#aec7e8", "#ffbb78",
    "#98df8a", "#ff9896", "#c5b0d5", "#c49c94", "#f7b6d2", "#9edae5",
    "#dbdb8d", "#c7c7c7", "#393b79", "#637939", "#8c6d31", "#843c39",
]

def build_per_saint_map(gender):
    """One folium map with one FeatureGroup per patron saint."""
    sub = long_df[(long_df["gender"] == gender) & long_df["latitude"].notna()].copy()
    if sub.empty:
        return None, 0, 0

    # Order saints by descending well count so frequent saints get
    # stable, distinguishable palette slots.
    order = (sub.drop_duplicates(subset=["item", "namedAfterLabel"])
                .groupby("namedAfterLabel").size()
                .sort_values(ascending=False).index.tolist())
    saint_colour = {name: PALETTE[i % len(PALETTE)] for i, name in enumerate(order)}

    centre = [sub["latitude"].mean(), sub["longitude"].mean()]
    m = folium.Map(location=centre, zoom_start=6, tiles="OpenStreetMap",
                   height=620)
    Fullscreen(position="topleft").add_to(m)

    # One FeatureGroup per saint. Swatch HTML in the name doubles as legend.
    groups = {}
    for saint in order:
        swatch = (f'<span style="display:inline-block;width:10px;height:10px;'
                  f'background:{saint_colour[saint]};border-radius:50%;'
                  f'margin-right:6px;vertical-align:middle"></span>')
        groups[saint] = folium.FeatureGroup(name=swatch + saint, show=True)
        m.add_child(groups[saint])

    for _, row in sub.iterrows():
        saint = row["namedAfterLabel"]
        colour = saint_colour[saint]
        popup_html = (
            f'<b>{row["itemLabel"]}</b><br>'
            f'<small>{row["country"]}</small><br>'
            f'<i>named after</i>: {saint}<br>'
            f'<small><a href="https://www.wikidata.org/wiki/{row["qid"]}" '
            f'target="_blank">Wikidata ({row["qid"]})</a></small>'
        )
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=5, color=colour, weight=1,
            fill=True, fill_color=colour, fill_opacity=0.8,
            popup=folium.Popup(popup_html, max_width=260),
        ).add_to(groups[saint])

    folium.LayerControl(collapsed=True).add_to(m)
    return m, len(sub), len(order)

map_female_saints, female_n, female_saint_count = build_per_saint_map("female")
print(f"  female: {female_n} markers across {female_saint_count} saints")
map_female_saints

### Step 3c — Male patrons, one layer per saint

Same construction as 3b, on the male subset. Patrick is typically
the most common, but the long tail of locally venerated saints
(Lachtain, Kieran, Declan, …) shows the finer-grained character of
a sacred landscape that blends national and very local figures.


In [ ]:
assert 'build_per_saint_map' in dir(), \
    "⚠ Please run the female per-saint map cell first (it defines the helper)."

map_male_saints, male_n, male_saint_count = build_per_saint_map("male")
print(f"  male: {male_n} markers across {male_saint_count} saints")
map_male_saints

### Step 3d — Top patrons per gender

The maps show *where*; the bar charts show *who*. One bar chart per
gender, Top-15 patrons each, wells counted once per patron per
well. Mary and Brigid dominate the female side by a wide margin —
their visual weight on the map is not just about geography but
about how often they are the *reason* a well exists at all.


In [ ]:
import matplotlib.pyplot as plt

assert 'long_df' in dir() and not long_df.empty, "⚠ Please run the data-loading cell first."

# Count wells per patron. Deduplicate on (item, namedAfter) so a well
# doesn't double-count if it somehow appears twice in the binding set.
dedup = long_df.drop_duplicates(subset=["item", "namedAfter"])

def top_patrons(gender, n=15):
    sub = dedup[dedup["gender"] == gender]
    return (sub.groupby("namedAfterLabel")
               .size()
               .sort_values(ascending=False)
               .head(n))

female_top = top_patrons("female")
male_top   = top_patrons("male")

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), sharex=False)

for ax, data, title, colour in [
    (axes[0], female_top, f"Top {len(female_top)} female patrons", "#c2185b"),
    (axes[1], male_top,   f"Top {len(male_top)} male patrons",     "#1565c0"),
]:
    ax.barh(data.index, data.values, color=colour)
    ax.set_xlabel("number of wells")
    ax.set_title(title)
    ax.invert_yaxis()
    for i, v in enumerate(data.values):
        ax.text(v + max(data.values) * 0.01, i, str(int(v)),
                va="center", fontsize=9)

plt.tight_layout()
plt.show()

## Step 4 — Explore

The two DataFrames — `long_df` (one row per well–patron pair) and
`wide_df` (one row per well) — stay in scope. Two quick
starting points:

### Gender ratio overall


In [ ]:
assert 'long_df' in dir(), "⚠ Please run the data-loading cell first."

dedup = long_df.drop_duplicates(subset=["item", "namedAfter"])
ratio = dedup["gender"].value_counts()
print("Well-patron pairs by gender:")
for g, n in ratio.items():
    pct = 100 * n / ratio.sum()
    print(f"  {g:6s}  {n:4d}   ({pct:5.1f}%)")

# How many *wells* are exclusively female-dedicated, male, or mixed?
assert 'wide_df' in dir()
def classify(genders):
    s = set(genders)
    if s == {"female"}: return "female only"
    if s == {"male"}:   return "male only"
    return "mixed"
wide_df["gender_class"] = wide_df["genders"].apply(classify)
print("\nWells by dedication class:")
for cls, n in wide_df["gender_class"].value_counts().items():
    print(f"  {cls:14s}  {n}")

### A specific saint on the map

Change `SAINT` below to any patron name that appears in the bar
chart (exact label, case-sensitive). The cell then prints all wells
dedicated to that saint with their coordinates and country.


In [ ]:
SAINT = "Brigid of Kildare"   # try: "Mary", "Winifred", "Saint Patrick", "Íte of Killeedy"

assert 'long_df' in dir(), "⚠ Please run the data-loading cell first."
sub = long_df[long_df["namedAfterLabel"] == SAINT]
print(f"{len(sub)} wells dedicated to {SAINT!r}:\n")
for _, row in sub.iterrows():
    print(f"  {row['itemLabel']:40s}  "
          f"({row['latitude']:.3f}, {row['longitude']:.3f})  "
          f"[{row['country']}]  {row['qid']}")

---

*Part of an Open Educational Resource series on knowledge graphs
and linked open data, produced in the context of
[NFDI4Objects](https://www.nfdi4objects.net/). Data: Wikidata
[WikiProject HolyWells](https://www.wikidata.org/wiki/Wikidata:WikiProject_HolyWells),
CC0. Tiles: OpenStreetMap contributors.*
